## Coding Attention mechanisms


### interesting topic :   Bahdanau attention
- self-attention mechanism inspired by the Bahdanau attention mechanism.
- Self-attention is a mechanism in transformers used to compute
more efficient input representations
- The “self” in self-attention
In self-attention, the “self” refers to the mechanism’s ability to compute attention
weights by relating different positions within a single input sequence. It assesses and
learns the relationships and dependencies between various parts of the input itself,
such as words in a sentence or pixels in an image.

### A simple self-attention mechanism without trainable weights

- ω,attention scores
- z context vector 
- alpha attention weight after normalization of attention score

In [2]:
import  torch
inputs =torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

In [3]:
inputs.shape

torch.Size([6, 3])

In [4]:
query = inputs[1] # Your (x^1)
attn_scores_2 = torch.empty(inputs.shape[0])

for i , x_i in enumerate(inputs):
    print(f"Calculating attention score for x^{i+1} with query x^2")
    print(f"Query: {query}")
    print(f"Key: {x_i}")
    attn_scores_2[i] = torch.dot(query,x_i)
    print(f"Attention score: {attn_scores_2[i]}")
    print("-"*50)

Calculating attention score for x^1 with query x^2
Query: tensor([0.5500, 0.8700, 0.6600])
Key: tensor([0.4300, 0.1500, 0.8900])
Attention score: 0.9544000625610352
--------------------------------------------------
Calculating attention score for x^2 with query x^2
Query: tensor([0.5500, 0.8700, 0.6600])
Key: tensor([0.5500, 0.8700, 0.6600])
Attention score: 1.4950001239776611
--------------------------------------------------
Calculating attention score for x^3 with query x^2
Query: tensor([0.5500, 0.8700, 0.6600])
Key: tensor([0.5700, 0.8500, 0.6400])
Attention score: 1.4754000902175903
--------------------------------------------------
Calculating attention score for x^4 with query x^2
Query: tensor([0.5500, 0.8700, 0.6600])
Key: tensor([0.2200, 0.5800, 0.3300])
Attention score: 0.8434000015258789
--------------------------------------------------
Calculating attention score for x^5 with query x^2
Query: tensor([0.5500, 0.8700, 0.6600])
Key: tensor([0.7700, 0.2500, 0.1000])
Attenti

- the dot product is a measure of similarity
because it quantifies how closely two vectors are aligned: a higher dot product indi-
cates a greater degree of alignment or similarity between the vectors.
 - This normalization is a convention that is useful for interpre-
tation and maintaining training stability in an LLM.

In [5]:
attn_weights_2_temp = attn_scores_2 / attn_scores_2.sum() # better to use softmax function for normalization but here we are using simple sum for normalization
print(f"Attention weights (unnormalized): {attn_scores_2}")
print(f"Attention weights (normalized): {attn_weights_2_temp}")

Attention weights (unnormalized): tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
Attention weights (normalized): tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])


In [6]:
def softmax_naive(x):
    exp_x = torch.exp(x)
    return exp_x / exp_x.sum(dim=0)


attn_weights_2_naive = softmax_naive(attn_scores_2)
print(f"Attention weights (softmax normalized): {attn_weights_2_naive}")
print(f"Sum of unnormalized attention weights: {attn_scores_2.sum()}")
print(f"Sum of normalized attention weights (simple sum): {attn_weights_2_temp.sum()}")
print(f"Sum of normalized attention weights (softmax): {attn_weights_2_naive.sum()}")

Attention weights (softmax normalized): tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum of unnormalized attention weights: 6.561699867248535
Sum of normalized attention weights (simple sum): 1.0000001192092896
Sum of normalized attention weights (softmax): 1.0


- softmax function ensures that the attention weights are always positive. 

- makes the output interpretable as probabilities or relative importance, where higher weights indicate greater importance

In [7]:
attn_weights_2 =torch.softmax(attn_scores_2, dim=0)
print(f"Attention weights (softmax normalized using PyTorch): {attn_weights_2}")
print(f"Sum of normalized attention weights (softmax using PyTorch): {attn_weights_2.sum()}")

Attention weights (softmax normalized using PyTorch): tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum of normalized attention weights (softmax using PyTorch): 1.0


In [8]:
query = inputs[1]# Your (x^1)
context_vector_2 = torch.zeros(query.shape) # initialize context vector with zeros
for i , x_i in enumerate(inputs):
    context_vector_2 += attn_weights_2[i]*x_i
    print(context_vector_2)
    
print("-"*50)
print(f"final context_vector: { context_vector_2}" )

tensor([0.0596, 0.0208, 0.1233])
tensor([0.1904, 0.2277, 0.2803])
tensor([0.3234, 0.4260, 0.4296])
tensor([0.3507, 0.4979, 0.4705])
tensor([0.4340, 0.5250, 0.4813])
tensor([0.4419, 0.6515, 0.5683])
--------------------------------------------------
final context_vector: tensor([0.4419, 0.6515, 0.5683])


### Computing attention weights for all input tokens

1. Compute the attention scores as
dot products between the inputs.

2. The attention weights are a normalized
version of the attention scores.

3. The context vectors are computed as
a weighted sum over the inputs.

In [9]:
attn_scores = torch.empty(6 ,6)
for i , x_i in enumerate(inputs):
    for j , x_j in enumerate(inputs):
        attn_scores[ i , j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [10]:
## for loops are generally slow, and we can achieve the same results using matrix multiplication

attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [11]:
attn_weights = torch.softmax( attn_scores , dim =-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [12]:
all_context_vectors = attn_weights @ inputs
print(all_context_vectors)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


### Scaled dot-product attention

- The scaling by the square root of the embedding dimension is the reason why this self-attention mechanism is also called scaled-dot product attention. why scaled because if we just take softmax our gradients might get near zero and learning process will be very slow 

In [13]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2



In [14]:
torch.manual_seed(123)

w_query = torch.nn.Parameter(torch.rand(d_in , d_out), requires_grad=False)
w_key = torch.nn.Parameter(torch.rand(d_in , d_out) , requires_grad=False)
w_value = torch.nn.Parameter(torch.rand(d_in, d_out) , requires_grad=False)

In [15]:
print(w_query)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])


In [16]:
print(x_2)

tensor([0.5500, 0.8700, 0.6600])


In [17]:
query_2 = x_2 @ w_query
key_2 =x_2 @ w_key
value_2 = x_2 @ w_value

print(query_2)

tensor([0.4306, 1.4551])


In [18]:
keys = inputs @ w_key
values = inputs @ w_value

print("Keys.shape:", keys.shape)
print("values.shape: ", values.shape) 


Keys.shape: torch.Size([6, 2])
values.shape:  torch.Size([6, 2])


In [19]:
keys_2 = keys[1]
attn_scores_22 =query_2.dot(keys_2)

In [20]:
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [21]:
d_k = keys.shape[-1]
print(d_k)

attn_weights_2 = torch.softmax(attn_scores_2/d_k**0.5 , dim =-1)
print(attn_weights_2)

2
tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [22]:
context_vector_2 = attn_weights_2 @ values
print(context_vector_2)

tensor([0.3061, 0.8210])


- Once the model determines which keys (and thus which parts of the input) are most relevant to the query (the current focus item), it retrieves the corresponding values.

- Each token searches the sentence for relevant tokens and aggregates their information.

- smart information mixing mechanism and become context ware representation
- language understanding
- long-range dependencies
- reasoning

### A compact self-attention class

In [23]:
print(d_in)

3


In [24]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self , d_in , d_out):
        super().__init__()
        self.w_query = nn.Parameter(torch.rand(d_in , d_out))
        self.w_key = nn.Parameter(torch.rand(d_in , d_out))
        self.w_value = nn.Parameter(torch.rand(d_in , d_out))
        
    
    def forward(self , x):
        queries = x @ self.w_query
        values = x@ self.w_value
        keys = x@ self.w_key
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5 , dim=-1)
        context_vec = attn_weights @ values 
        return context_vec

In [25]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in , d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [26]:
class SelfAttention_v2(nn.Module): 
    def __init__(self , d_in, d_out , qkv_bias=False):
         super().__init__()
         self.w_query = nn.Linear(d_in , d_out , bias = qkv_bias)
         self.w_key = nn.Linear(d_in , d_out , bias = qkv_bias)
         self.w_value = nn.Linear(d_in , d_out , bias = qkv_bias)
         
    def forward(self , x):
        queries =self.w_query(x)
        values =self.w_value(x)
        keys =  self.w_key(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5 , dim=-1)
        context_vec = attn_weights @ values 
        return context_vec
        

In [28]:
torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in , d_out)
print(sa_v1(inputs))

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


## 3. 5 Applying a casual attention mask
- casual attention means masked attention 
- masked future attention weight 
- nonmasked attention weights will sum up to 1 

In [29]:
queries = sa_v2.w_query(inputs)
keys = sa_v2.w_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5 , dim=-1)
print(attn_weights)

tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


In [30]:
context_length = inputs.shape[0]
print(context_length)

mask_simple = torch.tril(torch.ones(context_length , context_length))
print(mask_simple)

6
tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [31]:
masked_attn_scores = attn_weights * mask_simple
print(masked_attn_scores)

tensor([[0.1717, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1749, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1637, 0.1749, 0.1746, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.0000, 0.0000],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<MulBackward0>)


In [32]:
row_sums = masked_attn_scores.sum(dim=-1, keepdim=True)
masked_simple_norm =masked_attn_scores / row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<DivBackward0>)


###  Information leakage :
- because of renormalized our masked attention weight no information about future tokens is leaked

In [47]:
print(torch.triu(torch.ones(4, 4),diagonal=1) )

tensor([[0., 1., 1., 1.],
        [0., 0., 1., 1.],
        [0., 0., 0., 1.],
        [0., 0., 0., 0.]])


In [33]:
mask= torch.triu(torch.ones(context_length , context_length) , diagonal=1)
print(mask)
masked= attn_scores.masked_fill(mask.bool() , -torch.inf)
print(masked)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])
tensor([[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
        [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
        [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
        [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
        [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],
       grad_fn=<MaskedFillBackward0>)


In [43]:
attn_weights= torch.softmax(masked/keys.shape[-1]**0.5 , dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


In [51]:
context_vec= attn_weights @ values
print(context_vec)

tensor([[0.1855, 0.8812],
        [0.2938, 0.9445],
        [0.3258, 0.9576],
        [0.3036, 0.8564],
        [0.2737, 0.7564],
        [0.2818, 0.7599]], grad_fn=<MmBackward0>)


### Masking additional attention weights with dropout 
- dropout in the attention mechanism is typically applied at two specific times: after calculating the attention weights or after applying the attention weights to the value vectors. 

In [ ]:
torch.manual_seed(123)
dropout = nn.Dropout(0.5)
example = torch.ones(6, 6)
print(dropout(example))

## we are doing drop out  50% for this we are compensate 
# that by multiplying the remaining values by 2 (1/(1-0.5)) to 
# maintain the expected value of the inputs after dropout. 
# This technique is known as "inverted dropout" and helps to ensure 
# that the scale of the activations remains consistent during training, 
# even when some units are randomly dropped out.

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [54]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6380, 0.6816, 0.6804, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5090, 0.5085, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4120, 0.0000, 0.3869, 0.0000, 0.0000],
        [0.0000, 0.3418, 0.3413, 0.3308, 0.3249, 0.0000]],
       grad_fn=<MulBackward0>)


### Implement a compact casual attention class

In [55]:
batch =torch.stack((inputs , inputs), dim=0)
# batch of 2 sequences

print(batch.shape)

torch.Size([2, 6, 3])


In [57]:
class CausalSelfAttention(nn.Module):
    def __init__(self , d_in, d_out ,context_length, dropout , qkv_bias=False):
         super().__init__()
         self.d_out = d_out
         self.w_query = nn.Linear(d_in , d_out , bias = qkv_bias)
         self.w_key = nn.Linear(d_in , d_out , bias = qkv_bias)
         self.w_value = nn.Linear(d_in , d_out , bias = qkv_bias)
         self.dropout = nn.Dropout(dropout)
         self.register_buffer("mask" , 
                            torch.triu(torch.ones(context_length , context_length) , 
                            diagonal=1))
         
    def forward(self , x):
        b, num_tokens, d_in = x.shape
        queries =self.w_query(x)
        values =self.w_value(x)
        keys =  self.w_key(x)
        attn_scores = queries @ keys.transpose(1, 2)
        masked_attn_scores = attn_scores.masked_fill(
            self.mask.bool()[:num_tokens , :num_tokens] , -torch.inf)
        attn_weights = torch.softmax(masked_attn_scores/keys.shape[-1]**0.5 , dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ values 
        return context_vec

In [58]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalSelfAttention(d_in , d_out , context_length , dropout=0.0)
context_vecs= ca(batch)
print(context_vecs.shape)

torch.Size([2, 6, 2])


In [59]:
print(context_vecs)

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)


## 3.6 Extending Single Attention to Multi-head attention

### 3.6.1 Stacking multiple single-head attention layers

### 3.6.2 Implementing multi-head attention with weights splits